# Visualize Weka speed data

In [ ]:
# Imports
import json

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from dataclasses import dataclass, asdict
from pathlib import Path

from weka_helpers import RatioContainer, ResultMap, Generation, convert_results_to_primitive, load_results_from_json

In [ ]:
# Input parameters

input_dir = Path("/rubin/bench/weka")

input_files: list[Path] = input_dir.glob("*.json")

In [ ]:
type ResultMapDataframe = dict[str,list[int|float|str]]

def add_data_to_rmd(rmd: ResultMapDataframe, rm: ResultMap, generation: str) -> ResultMapDataframe:
    rmd["sizes"].extend(list(rm.sizes))
    rmd["start_times"].extend(list(rm.start_times))
    rmd["speeds_reported"].extend(list(rm.speeds_reported))
    rmd["generation"].extend([Generation(generation) for x in rm.sizes])
    return rmd

def ratio_container_to_dataframe(ratio_container: RatioContainer) -> pd.DataFrame:
    rmd = {
        "sizes": [],
        "start_times": [],
        "speeds_reported": [],
        "generation": [],
    }
    rmd=add_data_to_rmd(rmd, ratio_container.initial, generation="initial")
    if ratio_container.recheck:
        rmd=add_data_to_rmd(rmd, ratio_container.recheck, generation="recheck")
    return pd.DataFrame(data=rmd)
    
def plot_result(ratio_container: RatioContainer) -> None:
    plot_individual_speeds(ratio_container)
    

def plot_individual_speeds(ratio_container: RatioContainer) -> None:
    df = ratio_container_to_dataframe(ratio_container)
    print(df)
    fig=go.Figure()
    fig.add_trace(
        go.Scatter(
            x=df.start_times,
            y=df.speeds_reported,
            # color="generation"
        ),
    )
    fig.update_layout(
        title_text = "Weka S3 Gateway speed"
    )
    fig.show()
    print(df)

# Build a list of ResultMaps from all the files

In [ ]:
res: list[RatioContainer] = []
for fn in input_files:
    rms=load_results_from_json(fn)
    res.extend(rms)

for ratio_container in res:
    plot_result(ratio_container)